In [34]:
import pandas as pd
import numpy as np

import logging
import requests

from datetime import datetime
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv

import pymongo
from pymongo import MongoClient


In [35]:
# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


In [36]:
# Path setup
RAW       = Path("data/raw")
PROCESSED = Path("data/processed")
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# File paths
CALLS_RAW    = RAW / "911_calls.csv"
EVENTS_RAW   = RAW / "public_events.csv"
ENRICHED_OUT = PROCESSED / "enriched_calls.csv"

# MongoDB config
load_dotenv(find_dotenv(), override=True)
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME         = "austin_911"
COLLECTION_NAME = "calls"

log.info("Raw path       : %s", RAW)
log.info("Processed path : %s", PROCESSED)
log.info("Calls file exists: %s", CALLS_RAW.exists())
log.info("MongoDB target : %s > %s", DB_NAME, COLLECTION_NAME)

21:39:42  INFO      Raw path       : data/raw
21:39:42  INFO      Processed path : data/processed
21:39:42  INFO      Calls file exists: True
21:39:42  INFO      MongoDB target : austin_911 > calls


In [ ]:
# Load 911 calls
df_calls = pd.read_csv(RAW / "911_calls.csv")

log.info("Raw dataset shape: %s", df_calls.shape)
log.info("Columns: %s", list(df_calls.columns))

# Preview
df_calls.head()

21:39:52  INFO      Raw dataset shape: (3974470, 27)
21:39:52  INFO      Columns: ['Incident Number', 'Incident Type', 'Council District', 'Mental Health Flag', 'Priority Level', 'Response Datetime', 'Response Year', 'Response Month', 'Response Day of Week', 'Response Hour', 'First Unit Arrived Datetime', 'Call Closed Datetime', 'Sector', 'Initial Problem Description', 'Initial Problem Category', 'Final Problem Description', 'Final Problem Category', 'Number of Units Arrived', 'Unit Time on Scene', 'Call Disposition Description', 'Report Written Flag', 'Response Time', 'Officer Injured/Killed Count', 'Subject Injured/Killed Count', 'Other Injured/Killed Count', 'Geo ID', 'Census Block Group']


,Incident Number,Incident Type,Council District,Mental Health Flag,Priority Level,Response Datetime,Response Year,Response Month,Response Day of Week,Response Hour,...,Number of Units Arrived,Unit Time on Scene,Call Disposition Description,Report Written Flag,Response Time,Officer Injured/Killed Count,Subject Injured/Killed Count,Other Injured/Killed Count,Geo ID,Census Block Group
0,260530390,Dispatched Incident,1,Not Mental Health Incident,Priority 2,02/22/2026 06:26:02 AM,2026,Feb,Sun,6,...,2.0,9320.0,No Report,0,905.0,0,0,0,4.845304e+11,4.530440e+09
1,251700545,Dispatched Incident,10,Not Mental Health Incident,Priority 2,06/19/2025 11:25:05 AM,2025,Jun,Thu,11,...,1.0,1490.0,No Report,0,310.0,0,0,0,4.845303e+11,4.530306e+09
2,250401212,Dispatched Incident,2,Not Mental Health Incident,Priority 2,02/09/2025 08:56:39 PM,2025,Feb,Sun,20,...,2.0,2462.0,No Report,0,161.0,0,0,0,4.845398e+11,4.539800e+09
3,182151197,Dispatched Incident,7,Not Mental Health Incident,Priority 2,08/03/2018 03:24:53 PM,2018,Aug,Fri,15,...,2.0,5547.0,No Report,0,702.0,0,0,0,4.845304e+11,4.530440e+09
4,183160413,Officer-Initiated Incident,2,Not Mental Health Incident,Priority 3,11/12/2018 09:04:27 AM,2018,Nov,Mon,9,...,1.0,8.0,No Report,0,NaN,0,0,0,4.845398e+11,4.539800e+09


In [ ]:
# Filter down
df_clean = df_calls.dropna(subset=[
    'Response Datetime',
    'Priority Level',
    'Sector',
    'Initial Problem Category',
    'Response Time'
])

# 2024
df_clean = df_clean[df_clean['Response Year'] == 2024]

#10,000 rows
df_sample = df_clean.sample(n=10000, random_state=42).reset_index(drop=True)

log.info("Filtered shape : %s", df_clean.shape)
log.info("Sample shape   : %s", df_sample.shape)

df_sample.head()

21:39:52  INFO      Filtered shape : (286035, 27)
21:39:52  INFO      Sample shape   : (10000, 27)


,Incident Number,Incident Type,Council District,Mental Health Flag,Priority Level,Response Datetime,Response Year,Response Month,Response Day of Week,Response Hour,...,Number of Units Arrived,Unit Time on Scene,Call Disposition Description,Report Written Flag,Response Time,Officer Injured/Killed Count,Subject Injured/Killed Count,Other Injured/Killed Count,Geo ID,Census Block Group
0,242211074,Dispatched Incident,2,Not Mental Health Incident,Priority 3,08/08/2024 04:51:34 PM,2024,Aug,Thu,16,...,1.0,1462.0,No Report,0,3043.0,0,0,0,4.845300e+11,4.530024e+09
1,242510524,Dispatched Incident,1,Not Mental Health Incident,Priority 2,09/07/2024 09:40:44 AM,2024,Sep,Sat,9,...,2.0,941.0,No Report,0,1099.0,0,0,0,4.845304e+11,4.530438e+09
2,240070288,Dispatched Incident,7,Not Mental Health Incident,Priority 0,01/07/2024 05:14:30 AM,2024,Jan,Sun,5,...,2.0,6039.0,Report Written,1,859.0,0,0,0,4.845304e+11,4.530440e+09
3,243290318,Dispatched Incident,3,Not Mental Health Incident,Priority 0,11/24/2024 03:50:12 AM,2024,Nov,Sun,3,...,2.0,14271.0,No Report,0,358.0,0,0,0,4.845300e+11,4.530023e+09
4,242511364,Dispatched Incident,8,Not Mental Health Incident,Priority 3,09/07/2024 08:06:53 PM,2024,Sep,Sat,20,...,1.0,647.0,Non-Police Matter,0,9619.0,0,0,0,4.845300e+11,4.530019e+09


In [ ]:
# Check sample
log.info("Priority Level values: %s", df_sample['Priority Level'].value_counts().to_dict())
log.info("Top Incident Categories: %s", df_sample['Initial Problem Category'].value_counts().head(10).to_dict())
log.info("Sectors: %s", df_sample['Sector'].value_counts().to_dict())
log.info("Response Time stats:\n%s", df_sample['Response Time'].describe())

df_sample[['Response Datetime', 'Priority Level', 'Sector', 'Initial Problem Category', 'Response Time']].head(10)

21:39:52  INFO      Priority Level values: {'Priority 2': 4932, 'Priority 3': 1951, 'Priority 1': 1759, 'Priority 0': 1358}
21:39:52  INFO      Top Incident Categories: {'Disturbance': 1575, 'Trespassing': 1110, 'Welfare Check': 1083, 'Suspicious Things': 1056, 'Alarms': 1027, 'Crashes': 790, 'Traffic Stop/Hazard': 678, 'Other': 624, 'Assistance': 371, 'Disorderly Conduct': 355}
21:39:52  INFO      Sectors: {'Edward': 1404, 'David': 1364, 'Adam': 1182, 'Frank': 1142, 'Baker': 1098, 'Charlie': 1055, 'Henry': 1051, 'Ida': 967, 'George': 587, 'Airport': 150}
21:39:52  INFO      Response Time stats:
count    10000.000000
mean      2781.521200
std       5744.745842
min         20.000000
25%        462.000000
50%        927.000000
75%       2533.000000
max      88390.000000
Name: Response Time, dtype: float64


,Response Datetime,Priority Level,Sector,Initial Problem Category,Response Time
0,08/08/2024 04:51:34 PM,Priority 3,Frank,Welfare Check,3043.0
1,09/07/2024 09:40:44 AM,Priority 2,Edward,Disturbance,1099.0
2,01/07/2024 05:14:30 AM,Priority 0,Edward,Burglary,859.0
3,11/24/2024 03:50:12 AM,Priority 0,Henry,Traffic Stop/Hazard,358.0
4,09/07/2024 08:06:53 PM,Priority 3,David,Welfare Check,9619.0
5,08/02/2024 09:48:17 AM,Priority 1,Henry,Traffic Stop/Hazard,774.0
6,03/29/2024 09:30:46 PM,Priority 2,David,Disturbance,4523.0
7,01/17/2024 11:38:27 PM,Priority 2,Frank,Trespassing,815.0
8,12/14/2024 08:42:03 AM,Priority 0,Adam,Traffic Stop/Hazard,168.0
9,08/28/2024 05:13:17 PM,Priority 3,Adam,Auto Theft,15019.0


In [ ]:
# Datetime features
df_sample['Response Datetime'] = pd.to_datetime(df_sample['Response Datetime'])

df_sample['date']  = df_sample['Response Datetime'].dt.date
df_sample['hour']  = df_sample['Response Datetime'].dt.hour
df_sample['month'] = df_sample['Response Datetime'].dt.month
df_sample['day_of_week'] = df_sample['Response Datetime'].dt.day_name()
df_sample['is_weekend'] = df_sample['Response Datetime'].dt.dayofweek >= 5
df_sample['is_night'] = df_sample['hour'].between(20, 23) | df_sample['hour'].between(0, 5)

log.info("Date range: %s to %s", df_sample['date'].min(), df_sample['date'].max())
log.info("Sample datetime features:")
df_sample[['Response Datetime', 'date', 'hour', 'day_of_week', 'is_weekend', 'is_night']].head(5)

/var/folders/9y/d4rd096n1_sg8vb1q6qj23sw0000gn/T/ipykernel_34306/925702394.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_sample['Response Datetime'] = pd.to_datetime(df_sample['Response Datetime'])
21:39:53  INFO      Date range: 2024-01-01 to 2024-12-31
21:39:53  INFO      Sample datetime features:


,Response Datetime,date,hour,day_of_week,is_weekend,is_night
0,2024-08-08 16:51:34,2024-08-08,16,Thursday,False,False
1,2024-09-07 09:40:44,2024-09-07,9,Saturday,True,False
2,2024-01-07 05:14:30,2024-01-07,5,Sunday,True,True
3,2024-11-24 03:50:12,2024-11-24,3,Sunday,True,True
4,2024-09-07 20:06:53,2024-09-07,20,Saturday,True,True


In [41]:
# Connect to Atlas
client = MongoClient(MONGO_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Drop existing data if any
collection.drop()
log.info("Dropped existing collection")

21:39:53  INFO      Dropped existing collection


In [ ]:
records = df_sample.to_dict(orient='records')

for record in records:
    for key, value in record.items():
        if pd.isna(value) if not isinstance(value, (list, dict)) else False:
            record[key] = None
        elif hasattr(value, 'item'):  # numpy types
            record[key] = value.item()
        elif str(type(value)) == "<class 'datetime.date'>":
            record[key] = str(value)

# Insert
result = collection.insert_many(records)
log.info("Inserted %d documents into %s.%s", len(result.inserted_ids), DB_NAME, COLLECTION_NAME)

21:40:42  INFO      Inserted 10000 documents into austin_911.calls


In [44]:
# Verify
count = collection.count_documents({})
log.info("Total documents in collection: %d", count)

21:40:47  INFO      Total documents in collection: 10000
